In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from dataclasses import dataclass

@dataclass
class ColourContext:
    favourite_colour: str = "blue"
    least_favourite_colour: str = "yellow"

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model="mistral-small-latest",
    context_schema=ColourContext  
)

In [6]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

In [7]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='8664a0a0-8ee5-4635-8f11-baf53bb94601'),
              AIMessage(content='You haven’t told me your favorite color yet—so I can’t answer that! If you’d like, you can share it with me, and I’ll happily note it down. 😊', additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 21, 'total_tokens': 62, 'completion_tokens': 41, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small-latest', 'model': 'mistral-small-latest', 'finish_reason': 'stop', 'model_provider': 'mistralai'}, id='lc_run--019eedcf-9341-7480-8cef-c31969350b72-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 21, 'output_tokens': 41, 'total_tokens': 62})]}


## Accessing Context

In [8]:
from langchain.tools import tool, ToolRuntime

@tool
def get_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the favourite colour of the user"""
    return runtime.context.favourite_colour

@tool
def get_least_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the least favourite colour of the user"""
    return runtime.context.least_favourite_colour

In [9]:
agent = create_agent(
    model="mistral-small-latest",
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColourContext
)

In [10]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

pprint(response)

c:\Users\amrit\Documents\Projects\agent_experiments\.venv\Lib\site-packages\pydantic\functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='yellow'), input_type=ColourContext])
  function=lambda v, h: h(v), schema=original_schema
c:\Users\amrit\Documents\Projects\agent_experiments\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='yellow'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='44feb87d-d7f2-4508-83cd-9d41699dbef7'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'bAlLoGcUZ', 'type': 'function', 'function': {'name': 'get_favourite_colour', 'arguments': '{}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 117, 'total_tokens': 126, 'completion_tokens': 9, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small-latest', 'model': 'mistral-small-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019eedd0-aaa6-7470-970a-adc6092f3c10-0', tool_calls=[{'name': 'get_favourite_colour', 'args': {}, 'id': 'bAlLoGcUZ', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 117, 'output_tokens': 9, 'total_tokens': 126}),
              ToolMessage(content='blue', name='get_favourite_colour', id='00456893-2f88-43cd-a9ee-eccce4027551', to

In [11]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(favourite_colour="green")
)

pprint(response)

c:\Users\amrit\Documents\Projects\agent_experiments\.venv\Lib\site-packages\pydantic\functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='yellow'), input_type=ColourContext])
  function=lambda v, h: h(v), schema=original_schema
c:\Users\amrit\Documents\Projects\agent_experiments\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='yellow'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='fa06e4f3-2c8e-4ae8-aec6-ca718896c82d'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'rAvHGL7jD', 'type': 'function', 'function': {'name': 'get_favourite_colour', 'arguments': '{}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 117, 'total_tokens': 126, 'completion_tokens': 9, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small-latest', 'model': 'mistral-small-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019eedd0-b5ca-74b0-8e36-63a1c9bd6b9e-0', tool_calls=[{'name': 'get_favourite_colour', 'args': {}, 'id': 'rAvHGL7jD', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 117, 'output_tokens': 9, 'total_tokens': 126}),
              ToolMessage(content='green', name='get_favourite_colour', id='83ab0f90-b2b9-4898-b878-09eef5819b83', t

In [12]:
# THis context is immutable by agent, need state 